# 04 — Disjoint training regions from cyclic assignment

When the validation block sits between two runs of training blocks, the training set becomes spatially disjoint. `FoldPlanner._merge_adjacent` collapses adjacent training blocks into contiguous runs, and `SplitRegions.train` then holds either a single `CropRegion` or a list of them. This notebook confirms which folds yield a disjoint training set and visualises the resulting training runs.

Modules exercised: `pipelines.cross_validation_pipeline.folds.FoldPlanner`, `tools.regions.SplitRegions`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Count the training runs per fold

A fold whose training set splits into two runs is exactly the situation the inference stage warns about, because stitching requires one contiguous region per split.

In [ ]:
from configuration.cross_validation_config import CrossValidationConfig
from pipelines.cross_validation_pipeline.folds import FoldPlanner

config                     = CrossValidationConfig()
config.folds.n_folds       = 6
config.folds.azimuth_start = 0
config.folds.azimuth_end   = 600

planner = FoldPlanner(config, range_start=0, range_end=50)
plans   = planner.plans()

for plan in plans:
    train_regions = plan.split_regions.regions("train")
    n_runs        = len(train_regions)
    flag          = "DISJOINT" if n_runs > 1 else "contiguous"
    print(f"fold {plan.fold_index}: train_blocks={plan.train_blocks} -> {n_runs} run(s) [{flag}]")

## Cross-check against the raw merge of adjacent blocks

We call `_merge_adjacent` directly on each fold's training block list and confirm the number of merged runs matches the number of training regions in the plan.

In [ ]:
for plan in plans:
    merged    = planner._merge_adjacent(plan.train_blocks)
    n_regions = len(plan.split_regions.regions("train"))
    print(f"fold {plan.fold_index}: merged_runs={merged}  matches_plan={len(merged) == n_regions}")

## Visualise the training runs along the azimuth axis

Each fold is a row. Training runs are drawn as blue bands; the validation and test blocks are left blank. A fold with two separated blue bands is a disjoint training set.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.6 * len(plans) + 1))

for plan in plans:
    for region in plan.split_regions.regions("train"):
        ax.barh(plan.fold_index, region.azimuth_end - region.azimuth_start, left=region.azimuth_start, color="#4878a8", edgecolor="white")

ax.set_xlim(config.folds.azimuth_start, config.folds.azimuth_end)
ax.set_yticks(range(len(plans)))
ax.set_yticklabels([f"fold {i}" for i in range(len(plans))])
ax.invert_yaxis()
ax.set_xlabel("azimuth (samples)")
ax.set_title("Training runs per fold (blue bands)")
plt.show()

## Expected visual outcome

Folds whose validation block falls in the interior of the training blocks show two separated blue bands (a disjoint training set), while the others show one continuous band. The direct `_merge_adjacent` cross-check reports `matches_plan = True` for every fold.